# Anchor-Token Predictivity Replication: Gemma-2-2B-it

**Project context.** This notebook is a precursor to the value-conflict probing study. Before
building the conflict battery, I want to validate the *measurement point*.

The Anthropic emotions paper ("Emotion Concepts and their Function in a Large Language Model,"
Transformer Circuits, Apr 2026) measured emotion-vector activations at the token immediately
preceding the Assistant's response (the ":" after "Assistant" in their template) and reported
that activations there **predict activations across the subsequent response**. This
conflict-state design assumes the same property holds in the chosen open-weights subject model,
that the pre-response anchor is a valid fixed measurement point. This notebook tests that assumption
directly.

**Hypothesis (H1).** For emotion vector $v_e$ extracted from Gemma-2-2B-it, the projection of the
residual stream onto $v_e$ at the **anchor token** (final token of the formatted prompt, before any
response exists) predicts the **mean projection across the response tokens**, across a battery of
emotionally varied prompts.

**Success criterion.** Median Pearson *r* across emotions at the focus layer exceeds the 95th
percentile of a shuffled-anchor null distribution.

**Why this matters for the conflict project.** If H1 holds, the colon-equivalent position is licensed as
the measurement point for the conflict probe (Question-1 "state" test: decodable pre-output,
no response-token confounds). If H1 fails here, the conflict design needs a different measurement
strategy *before* any conflict data is collected.

**Hardware assumptions.** Single 16 GB GPU (RTX 5080 laptop), 64 GB system RAM.
Gemma-2-2B-it in bf16 ≈ 5.5 GB weights, leaving ample headroom for full-layer caching.

**Deliberate deviations from the paper** (documented inline as ⚠️ DEVIATION):
mini emotion set (12 vs 171), fewer stories per emotion, fraction-based story-token cutoff
(Anthropic's stories were longer), neutral-PC confound removal optional and off by default in v0.


## 0. Environment

Blackwell (RTX 50-series) note: requires a recent PyTorch built against CUDA 12.8+.
If you see *"no kernel image is available for execution on the device"*, your torch/CUDA build
predates Blackwell. Reinstall torch from the cu128 (or newer) index before debugging anything else.

Gemma is a **gated** model on Hugging Face. Accept the license at
https://huggingface.co/google/gemma-2-2b-it and authenticate below.


In [ ]:
# Hugging Face auth (Gemma is gated). Either:
#   1) set HF_TOKEN in your environment before launching jupyter
# pip install python-dotenv  (one-time)
from dotenv import load_dotenv
load_dotenv() 
#   2) uncomment the interactive login below.
# from huggingface_hub import notebook_login; notebook_login()
# from huggingface_hub import whoami
# try:
#     print("HF auth OK:", whoami()["name"])
# except Exception as e:
#     print("Not authenticated with Hugging Face. Gemma download will fail.", e)


In [ ]:
# Run once per environment, then comment out.
# %pip install -q -U transformer_lens transformers accelerate huggingface_hub pandas scikit-learn matplotlib tqdm
# For Blackwell GPUs, if torch lacks kernels for your card:
# %pip install -q -U torch --index-url https://download.pytorch.org/whl/cu128


In [ ]:
import gc, json, math, os, random, time
import torch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

SEED = 23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), "CUDA not available — check driver / torch build."
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

DATA = Path("data"); DATA.mkdir(exist_ok=True)
OUT  = Path("results"); OUT.mkdir(exist_ok=True)


## 1. Configuration

All experiment knobs in one place. `FOCUS_LAYER` ≈ two-thirds depth, following the paper's choice;
the layer sweep in §7 checks robustness of that choice rather than assuming it transfers.


In [ ]:
CFG = dict(
    model_name      = "gemma-2-2b-it",     # TransformerLens alias for google/gemma-2-2b-it
    n_layers        = 26,                   # gemma-2-2b
    d_model         = 2304,
    sweep_layers    = list(range(2, 26, 2)),# even layers; skip 0-1 (embedding-adjacent)
    focus_layer     = 17,                   # ~2/3 of 26
    # --- emotion vector extraction (mini battery) ---
    emotions        = ["happy","sad","angry","afraid","calm","desperate",
                       "guilty","proud","surprised","anxious","loving","bored"],
    n_stories_per_emotion = 30,             # ⚠️ DEVIATION: paper used 100 topics x 12 stories
    story_max_new_tokens  = 220,
    story_temperature     = 1.0,
    story_start_frac      = 0.25,           # ⚠️ DEVIATION: paper used absolute token 50;
                                            # our stories are shorter, so fraction-based
    # --- evaluation battery ---
    n_rollouts      = 5,
    max_new_tokens  = 150,
    gen_temperature = 0.8,
)
print(json.dumps({k:(v if not isinstance(v,list) or len(v)<15 else f"<{len(v)} items>")
                  for k,v in CFG.items()}, indent=2))


## 2. Load model

bf16, eval mode, no gradients anywhere in this notebook.


In [ ]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(
    CFG["model_name"], dtype=torch.bfloat16, device="cuda",
)
model.eval()
torch.set_grad_enabled(False)

print(f"Loaded {CFG['model_name']}: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")
assert model.cfg.n_layers == CFG["n_layers"] and model.cfg.d_model == CFG["d_model"]
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 3. Chat template and anchor identification

The anchor is the **final token of the formatted prompt** (Gemma's functional equivalent of the
paper's "Assistant:" colon). For Gemma-2 the template ends `<start_of_turn>model\n`, so the anchor
is the newline token after the `model` role header.

Two classic silent bugs guarded against here:

1. **Double-BOS.** `apply_chat_template` already emits `<bos>`. TransformerLens's `to_tokens`
   *also* prepends BOS by default. Always pass `prepend_bos=False` for templated strings.
2. **Off-by-one anchoring.** Locate the anchor by *decoding* the final tokens and asserting the
   expected structure, never by assuming an index.


In [ ]:
def format_chat(user_msg: str) -> str:
    msgs = [{"role": "user", "content": user_msg}]
    return model.tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )

def to_toks(s: str) -> torch.Tensor:
    # Templated strings already contain <bos>. Do NOT let TL add another.
    return model.to_tokens(s, prepend_bos=False)

# Template Structure
demo = format_chat("Hello!")
print(repr(demo))
demo_toks = to_toks(demo)
tail = [model.tokenizer.decode(t) for t in demo_toks[0, -6:]]
print("last 6 tokens:", tail)

assert demo.endswith("<start_of_turn>model\n"), "Unexpected template tail — inspect before proceeding."
bos_id = model.tokenizer.bos_token_id
assert (demo_toks[0] == bos_id).sum().item() == 1, "BOS appears != 1 time — double-BOS bug?"

def anchor_index(prompt_toks: torch.Tensor) -> int:
    """Index of the anchor = final token of the formatted prompt."""
    return prompt_toks.shape[1] - 1

print("anchor token decodes to:", repr(model.tokenizer.decode(demo_toks[0, anchor_index(demo_toks)])))


## 4. Mini emotion-story dataset

I follow the paper's appendix prompt, creating stories in which a character experiences the target
emotion, **never naming it or near-synonyms**. The emotion is conveyed only through action, body
language, dialogue, thoughts, and context. This prevents the extracted vectors from latching onto
emotion vocabulary.

Two paths:
- **Option A (preferred for quality):** Generate the stories with a stronger model
  (Claude, GPT, a hosted Llama-70B) and drop them in `data/stories.json` as
  `{emotion: [story, ...]}`. Extraction happens on *experiment's* model's activations over the text,
  so the generator's identity doesn't contaminate the vectors.
- **Option B (self-contained fallback, used below if no file found):** Generate with
  Gemma-2-2B-it itself. Quality is lower. Expect occasional rule violations. I have added the filter
  below to drop stories that name the emotion.

⏱ Option B: ~360 generations × ~220 tokens. Saves incrementally.


In [ ]:
STORY_TOPICS = [
    "A neighbor wants to install a fence",
    "An employee is asked to train their replacement",
    "A traveler's flight is delayed, causing them to miss an important event",
    "A student is accused of plagiarism",
    "Two friends both apply for the same job",
    "A person runs into their ex at a mutual friend's wedding",
    "Someone finds their grandmother's engagement ring in a pawn shop",
    "An adult child moves back in with their parents",
    "A person's car is towed from their own driveway",
    "Someone receives a friend request from a childhood bully",
]

STORY_PROMPT = (
    "Write one short story (a single paragraph, 120-180 words) based on the premise below.\n"
    "Topic: {topic}\n"
    "The story should follow a character who is feeling {emotion}.\n"
    "IMPORTANT: You must NEVER use the word '{emotion}' or any direct synonyms of it. "
    "Convey the emotion ONLY through the character's actions and behaviors, physical "
    "sensations and body language, dialogue and tone of voice, thoughts and internal "
    "reactions, and situational context. Output only the story text."
)

# crude synonym lists for the leakage filter (extend as needed)
LEAK = {
    "happy": ["happy","joy","delight","cheerful","glad"],
    "sad": ["sad","sorrow","grief","unhappy"],
    "angry": ["angry","anger","furious","rage","mad"],
    "afraid": ["afraid","fear","scared","terrified","frighten"],
    "calm": ["calm","serene","tranquil","peaceful"],
    "desperate": ["desperate","desperation"],
    "guilty": ["guilty","guilt"],
    "proud": ["proud","pride"],
    "surprised": ["surprised","surprise","astonish","shock"],
    "anxious": ["anxious","anxiety","nervous"],
    "loving": ["loving","love","affection"],
    "bored": ["bored","boredom","tedious"],
}

def leaks(emotion, text):
    t = text.lower()
    return any(w in t for w in LEAK[emotion])


In [ ]:
STORIES_PATH = DATA / "stories_seed.json"

if STORIES_PATH.exists():
    stories = json.loads(STORIES_PATH.read_text())
    print("Loaded stories.json:", {e: len(v) for e, v in stories.items()})
else:
    print("No data/stories.json found — generating with subject model (Option B).")
    stories = {e: [] for e in CFG["emotions"]}
    per_topic = math.ceil(CFG["n_stories_per_emotion"] / len(STORY_TOPICS))
    pbar = tqdm(total=len(CFG["emotions"]) * len(STORY_TOPICS) * per_topic)
    for emotion in CFG["emotions"]:
        for topic in STORY_TOPICS:
            for k in range(per_topic):
                if len(stories[emotion]) >= CFG["n_stories_per_emotion"]: 
                    pbar.update(1); continue
                prompt = format_chat(STORY_PROMPT.format(topic=topic, emotion=emotion))
                toks = to_toks(prompt)
                out = model.generate(
                    toks, max_new_tokens=CFG["story_max_new_tokens"],
                    temperature=CFG["story_temperature"], do_sample=True,
                    verbose=False, prepend_bos=False,
                )
                text = model.tokenizer.decode(out[0, toks.shape[1]:], skip_special_tokens=True).strip()
                if len(text.split()) >= 60 and not leaks(emotion, text):
                    stories[emotion].append(text)
                pbar.update(1)
        STORIES_PATH.write_text(json.dumps(stories, indent=1))  # incremental save
    pbar.close()
    print({e: len(v) for e, v in stories.items()})


## 5. Extract emotion vectors

Paper recipe, miniaturized: per story, mean residual-stream activation (`resid_post`) over tokens
from `story_start_frac` of the way in (emotion should be established by then) to the end;
per-emotion mean across stories; subtract the grand mean across emotions.

Vectors are kept **per layer** for the sweep. Unit-normalized copies are used for projections.

⚠️ DEVIATION (off in v0): The paper also projected out top PCs of emotionally *neutral* transcripts
to strip dataset confounds. I will add that in v1 if anchor/response correlations look suspiciously
uniform across emotions. Uniformity is the signature of a shared confound direction.


In [ ]:
def story_mean_resid(text: str) -> dict:
    toks = model.to_tokens(text, prepend_bos=True)   # raw text -> TL adds BOS (correct here)
    if toks.shape[1] > 400: toks = toks[:, :400]
    _, cache = model.run_with_cache(
        toks, names_filter=lambda n: n.endswith("hook_resid_post"))
    start = max(8, int(toks.shape[1] * CFG["story_start_frac"]))
    means = {l: cache["resid_post", l][0, start:].mean(0).float().cpu()
             for l in CFG["sweep_layers"]}
    del cache
    return means

# Accumulate
per_emotion = {e: {l: [] for l in CFG["sweep_layers"]} for e in CFG["emotions"]}
for e in tqdm(CFG["emotions"], desc="emotions"):
    for s in stories[e]:
        m = story_mean_resid(s)
        for l in CFG["sweep_layers"]:
            per_emotion[e][l].append(m[l])
torch.cuda.empty_cache()

emotion_mean = {e: {l: torch.stack(v).mean(0) for l, v in d.items()}
                for e, d in per_emotion.items()}
grand = {l: torch.stack([emotion_mean[e][l] for e in CFG["emotions"]]).mean(0)
         for l in CFG["sweep_layers"]}

emotion_vec = {e: {l: emotion_mean[e][l] - grand[l] for l in CFG["sweep_layers"]}
               for e in CFG["emotions"]}
emotion_dir = {e: {l: v / v.norm() for l, v in d.items()} for e, d in emotion_vec.items()}

torch.save({ "vec": emotion_vec, "cfg": CFG }, OUT / "emotion_vectors.pt")
print("saved", OUT / "emotion_vectors.pt")


In [ ]:
# Sanity check 1: Vector geometry should be non-degenerate and weakly human-plausible.
L = CFG["focus_layer"]
E = CFG["emotions"]
M = torch.stack([emotion_dir[e][L] for e in E])
sim = (M @ M.T).numpy()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(sim, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(E)), E, rotation=90); ax.set_yticks(range(len(E)), E)
fig.colorbar(im); ax.set_title(f"Emotion vector cosine similarity (layer {L})")
plt.tight_layout(); plt.show()
# Expect: afraid~anxious positive; happy vs sad negative; nothing pinned at ±1.


## 6. Evaluation battery and rollouts

Prompts with varied *implied* emotional content (None names an emotion).
For each prompt: Tokenize once, record the anchor index, sample `n_rollouts` responses.

Key structural fact exploited later: For a fixed prompt, **anchor activations are identical across
rollouts** (deterministic forward pass over the same prefix; sampling begins after the anchor).
Asserted explicitly in §7. If that assertion fails, something is wrong with the pipeline.


In [ ]:
EVAL_PROMPTS = [
    # implied positive
    "I just found out I passed my licensing exam on the first try. I've been studying for two years.",
    "My daughter took her first steps today and I caught it on video.",
    "We finally paid off the last of our debt this morning.",
    "My short story is going to be published in a real literary magazine.",
    "After five years apart, my best friend is moving back to my city next month.",
    "I adopted a senior dog from the shelter this weekend and he's asleep on my feet right now.",
    # implied negative / loss
    "My childhood home was sold today. The new owners are tearing it down.",
    "I have to give a speech at my grandfather's memorial and I don't know how to start.",
    "My landlord just gave us sixty days to move out and rents have doubled in this area.",
    "I sent my manuscript to twelve agents and got twelve rejections.",
    "We had to cancel the wedding. I don't really want to talk about why.",
    "My cat has been missing for three days. The shelter hasn't seen him.",
    # implied threat / urgency
    "There's a wildfire two ridges over and the evacuation map keeps updating toward us.",
    "I think someone has been logging into my email. The recovery phone number was changed.",
    "My car started making a grinding noise on the highway and the brake light just came on.",
    "The biopsy results come back tomorrow morning and I can't sleep.",
    # frustration / task
    "This is the fourth time the airline has rescheduled my flight this week.",
    "I've explained the same requirement to this contractor three times and the work is still wrong.",
    # neutral / informational (control)
    "Can you explain how a heat pump works?",
    "What's the difference between a 401k and an IRA?",
    "Summarize the rules of cricket for someone who has never seen a match.",
    "How do I convert a CSV file to JSON in Python?",
    "What are the main differences between TCP and UDP?",
    "Explain how vaccines produce immunity.",
]
print(len(EVAL_PROMPTS), "prompts x", CFG["n_rollouts"], "rollouts")


In [ ]:
TRANSCRIPTS_PATH = DATA / "transcripts.json"

if TRANSCRIPTS_PATH.exists():
    transcripts = json.loads(TRANSCRIPTS_PATH.read_text())
    print("Loaded", len(transcripts), "transcripts.")
else:
    transcripts = []  # rows: {prompt_id, rollout, prompt_text, full_text, prompt_len}
    for pid, p in enumerate(tqdm(EVAL_PROMPTS, desc="prompts")):
        chat = format_chat(p)
        toks = to_toks(chat)
        plen = toks.shape[1]
        for r in range(CFG["n_rollouts"]):
            out = model.generate(
                toks, max_new_tokens=CFG["max_new_tokens"],
                temperature=CFG["gen_temperature"], do_sample=True,
                verbose=False, prepend_bos=False,
            )
            transcripts.append(dict(
                prompt_id=pid, rollout=r, prompt_text=p,
                full_text=model.tokenizer.decode(out[0], skip_special_tokens=False),
                prompt_len=plen,
            ))
        TRANSCRIPTS_PATH.write_text(json.dumps(transcripts))
    print("Generated", len(transcripts), "transcripts.")


## 7. Measurement: anchor vs response-wide projections

For each transcript: one forward pass with caching; project residual stream onto each unit emotion
direction; record the projection **at the anchor** (position `prompt_len - 1`) and the **mean over
response tokens** (positions `prompt_len:` minus trailing specials), at every sweep layer.


In [ ]:
rows = []
anchor_check = {}

for t in tqdm(transcripts, desc="measuring"):
    toks = to_toks(t["full_text"])
    _, cache = model.run_with_cache(
        toks, names_filter=lambda n: n.endswith("hook_resid_post"))
    a_idx = t["prompt_len"] - 1
    for l in CFG["sweep_layers"]:
        resid = cache["resid_post", l][0].float().cpu()   # [seq, d_model]
        resp = resid[t["prompt_len"]:]
        for e in CFG["emotions"]:
            d = emotion_dir[e][l]
            rows.append(dict(
                prompt_id=t["prompt_id"], rollout=t["rollout"], layer=l, emotion=e,
                anchor=float(resid[a_idx] @ d),
                resp_mean=float((resp @ d).mean()),
            ))
    # Sanity Check: Anchor resid identical across rollouts of same prompt (focus layer).
    key = t["prompt_id"]
    a_vec = cache["resid_post", CFG["focus_layer"]][0, a_idx].float().cpu()
    if key in anchor_check:
        assert torch.allclose(anchor_check[key], a_vec, atol=1e-3), \
            "Anchor activations differ across rollouts — pipeline bug (check tokenization)."
    else:
        anchor_check[key] = a_vec
    del cache

df = pd.DataFrame(rows)
df.to_csv(OUT / "projections.csv", index=False)
print(df.shape, "-> results/projections.csv")
df.head()


## 8. Analysis

**Primary test.** Per emotion, at the focus layer: Pearson *r* between anchor projection (one value
per prompt) and response-mean projection (averaged over rollouts per prompt), across prompts.

**Null.** Shuffle anchors across prompts within each emotion (1,000 permutations); compare the
observed median-*r*-across-emotions against the null's 95th percentile.


In [ ]:
L = CFG["focus_layer"]
focus = (df[df.layer == L]
         .groupby(["prompt_id", "emotion"])
         .agg(anchor=("anchor", "first"), resp=("resp_mean", "mean"))
         .reset_index())

def corr_per_emotion(d):
    return d.groupby("emotion").apply(
        lambda g: np.corrcoef(g.anchor, g.resp)[0, 1], include_groups=False)

obs = corr_per_emotion(focus)
print("Observed r per emotion (layer", L, "):\n", obs.round(3).sort_values(ascending=False))
print("\nMedian r:", round(obs.median(), 3))

rng = np.random.default_rng(SEED)
null_medians = []
for _ in range(1000):
    shuf = focus.copy()
    shuf["anchor"] = (shuf.groupby("emotion")["anchor"]
                      .transform(lambda s: rng.permutation(s.values)))
    null_medians.append(corr_per_emotion(shuf).median())
null_medians = np.array(null_medians)
p95 = np.quantile(null_medians, 0.95)
print(f"\nNull median-r 95th pct: {p95:.3f}")
print("H1", "SUPPORTED" if obs.median() > p95 else "NOT supported", "at the focus layer.")


In [ ]:
# Visuals: per-emotion r bar chart + 2 example scatters + layer sweep heatmap.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

obs.sort_values().plot.barh(ax=axes[0])
axes[0].axvline(p95, ls="--", c="gray"); axes[0].set_title(f"r by emotion (layer {L}); dashed = null 95th pct")

for ax, e in zip(axes[1:], ["desperate", "happy"]):
    g = focus[focus.emotion == e]
    ax.scatter(g.anchor, g.resp, s=18)
    ax.set_xlabel("anchor proj"); ax.set_ylabel("response mean proj")
    ax.set_title(f"{e}: r={np.corrcoef(g.anchor, g.resp)[0,1]:.2f}")
plt.tight_layout(); plt.show()

sweep = (df.groupby(["layer", "prompt_id", "emotion"])
           .agg(anchor=("anchor","first"), resp=("resp_mean","mean")).reset_index()
           .groupby(["layer","emotion"])
           .apply(lambda g: np.corrcoef(g.anchor,g.resp)[0,1], include_groups=False)
           .unstack())
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(sweep.values, aspect="auto", vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_yticks(range(len(sweep.index)), sweep.index)
ax.set_xticks(range(len(sweep.columns)), sweep.columns, rotation=90)
ax.set_ylabel("layer"); fig.colorbar(im); ax.set_title("anchor->response r, layer x emotion")
plt.tight_layout(); plt.show()


## 9. Controls and diagnostics

1. **Cross-prompt shuffle** is the primary null (above).
2. **Anchor-token outlier check**
   Boundary/special tokens can be attention sinks with outlier norms.
   If the anchor's residual norm is wildly atypical, projections there may be
   dominated by a few rogue dimensions, and mean over last 3 prompt tokens
   is the fallback.
3. **Neutral-prompt behavior**
   The six neutral controls should show muted anchor projections
   across emotive directions. Tf they don't, suspect a confound
   direction (see §5 deviation note).


In [ ]:
# Anchor norm vs typical positions (one transcript per prompt, focus layer)
norms_anchor, norms_other = [], []
for t in transcripts[:: CFG["n_rollouts"]]:
    toks = to_toks(t["full_text"])
    _, cache = model.run_with_cache(toks, names_filter=lambda n: n.endswith(f"blocks.{L}.hook_resid_post"))
    r = cache["resid_post", L][0].float()
    norms_anchor.append(r[t["prompt_len"]-1].norm().item())
    norms_other.append(r[5:t["prompt_len"]-1].norm(dim=-1).median().item())
    del cache
print(f"anchor norm median {np.median(norms_anchor):.1f} | other-position median {np.median(norms_other):.1f}")
print("ratio:", round(np.median(norms_anchor)/np.median(norms_other), 2),
      "(if >> 2-3x, consider averaging the last 3 prompt tokens as the anchor)")


## 10. Findings log

**Run date / environment:**

**H1 verdict at focus layer:**  median r = ____ vs null 95th pct = ____

**Layer profile:** where does predictivity peak? Does ~2/3 depth hold for Gemma-2-2B, or does the
focus layer need revising before the conflict study inherits it?

**Per-emotion pattern:** which emotions predict well/poorly? (Paper prior: concept vectors are
broadly predictive; a *uniform* r across all emotions would be suspicious — see confound note §5.)

**Anchor norm diagnostic:** outlier? measurement fallback needed?

**Story quality (Option B only):** leakage-filter rejection rate; obviously degenerate stories?
Decision: regenerate with stronger model for v1?

**Deviations that mattered:**

**Decision for scale-up:** proceed to Llama-3.1-8B replication ☐ / revise design first ☐
(8B notes: bf16 is ~16 GB — selective layer caching only, batch=1, generate with HF then re-forward;
or run on CMU cluster.)

**Implications for the conflict battery:** is the anchor licensed as the measurement point?
What changes, if anything, in the §3 anchoring procedure?

**Open questions / surprises:**


## Appendix: memory hygiene

If you hit OOM mid-run: `del cache`, `gc.collect()`, `torch.cuda.empty_cache()`, restart kernel as a
last resort (artifacts are saved incrementally to `data/` and `results/`, so reruns resume cheaply).


In [ ]:
def vram():
    print(f"allocated {torch.cuda.memory_allocated()/1e9:.2f} GB | reserved {torch.cuda.memory_reserved()/1e9:.2f} GB")
vram()
